In [1]:
import pandas as pd
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
import os
import sys
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data, write_to_hdf5_file, get_metadata
import seaborn as sns

In [2]:
Xy = pd.read_csv('icu_sleeplab_Xy.csv')
Xy = Xy.drop_duplicates(subset=['study_id'])

# Xy = pd.read_csv('icu_sleeplab_Xy_nights3.csv')

In [3]:
print(Xy.shape)
Xy = Xy.loc[np.logical_not(pd.isna(Xy.IBI_mean)),:]
print(Xy.shape)

(545, 671)
(535, 671)


In [4]:
Xy.head()

,study_id,night_id,population,MRN,Age,Sex,IBI_mean,IBI_std,IBI_median,IBI_Q75,...,ArousalIndexSSComb1,h_data_SSExpert,h_sleep_SSExpert,N1SSExpert,N2SSExpert,N3SSExpert,RSSExpert,SFISSExpert,SFI2SSExpert,ArousalIndexSSExpert
0,icu_001,2018-06-05,icu,1298742.0,79.0,Male,7.958415,5.441516,5.500000,15.000000,...,7714.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,icu_003,2018-06-18,icu,1910753.0,84.0,Male,5.531713,4.547764,3.900391,6.000000,...,4941.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,icu_004,2018-06-19,icu,6188100.0,32.0,Female,8.970362,5.474474,8.550781,15.000000,...,1038.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,icu_011,2018-09-19,icu,2770155.0,56.0,Female,5.070766,3.533123,4.101562,5.398438,...,10.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,icu_012,2018-09-24,icu,3826907.0,77.0,Female,10.533203,7.736719,15.000000,15.000000,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
q75s = [x for x in Xy.columns if 'Q75' in x]
for q75 in q75s:
    q25 = q75.replace('75','25')
    iqr = q75.replace('Q75','IQR')
    Xy[iqr] = Xy[q75] - Xy[q25]

In [6]:

for ss_v in ['SSResp', 'SSComb1', 'SSExpert']:
    Xy[f'N2N3{ss_v}'] = Xy[f'N2{ss_v}'] + Xy[f'N3{ss_v}']
    
    
ss_vs = ['SSResp_', 'SSComb1_', 'SSExpert_']
resp_vs = ['IBI', 'RR', 'MV', 'BSI']


for ss_v in ss_vs:
    for resp_v in resp_vs:
        Xy[resp_v + '_mean_' + ss_v + 'S'] = np.nanmean(np.array([Xy[resp_v + '_mean_' + ss_v + 'N1'].values, Xy[resp_v + '_mean_' + ss_v + 'N2'].values, 
                                                     Xy[resp_v + '_mean_' + ss_v + 'N3'].values, Xy[resp_v + '_mean_' + ss_v + 'R'].values]), axis=0)
        Xy[resp_v + '_std_' + ss_v + 'S'] = np.nanmean(np.array([Xy[resp_v + '_std_' + ss_v + 'N1'].values, Xy[resp_v + '_std_' + ss_v + 'N2'].values, 
                                                     Xy[resp_v + '_std_' + ss_v + 'N3'].values, Xy[resp_v + '_std_' + ss_v + 'R'].values]) , axis=0  ) 
        Xy[resp_v + '_median_' + ss_v + 'S'] = np.nanmean(np.array([Xy[resp_v + '_median_' + ss_v + 'N1'].values, Xy[resp_v + '_median_' + ss_v + 'N2'].values, 
                                                     Xy[resp_v + '_mean_' + ss_v + 'N3'].values, Xy[resp_v + '_mean_' + ss_v + 'R'].values]), axis=0)
        Xy[resp_v + '_IQR_' + ss_v + 'S'] = np.nanmean(np.array([Xy[resp_v + '_IQR_' + ss_v + 'N1'].values, Xy[resp_v + '_IQR_' + ss_v + 'N2'].values, 
                                                     Xy[resp_v + '_IQR_' + ss_v + 'N3'].values, Xy[resp_v + '_IQR_' + ss_v + 'R'].values]) , axis=0  ) 

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:12: RuntimeWarning: Mean of empty slice
  if sys.path[0] == '':
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:14: RuntimeWarning: Mean of empty slice
  
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:16: RuntimeWarning: Mean of empty slice
  app.launch_new_instance()
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:18: RuntimeWarning: Mean of empty slice


In [7]:
matching = pd.read_csv('Matched_Cohort_ICU_Sleeplab_withAirgo.csv')
matching['study_id'] = matching['study_id'].apply(lambda x: x.replace('Air', 'sleeplab_').replace('ICU', 'icu_'))

In [8]:
matching.head(3)

,Age,MRN,Population,Sex,study_id,Sex_int
0,81.9,183939.0,Sleeplab,Male,sleeplab_303,True
1,78.6,405144.0,Sleeplab,Male,sleeplab_087,True
2,75.3,658243.0,Sleeplab,Male,sleeplab_448,True


In [9]:
Xy_icu = Xy[Xy.population=='icu']
Xy_sleeplab = Xy[Xy.population=='sleeplab']

# only used matched patients:
Xy_sleeplab = Xy_sleeplab.loc[np.isin(Xy_sleeplab.study_id, matching.study_id),:]

# so far, we have computed the statistics for icu patients for full stay, not split up into nights. therefore remove, duplicate rows for them:
# Xy.drop_duplicates(subset=['study_id'], inplace=True)

print(Xy.shape)
print(Xy_sleeplab.shape)
print(Xy_icu.shape)
print(len(pd.unique(Xy_icu.study_id)))

(535, 810)
(242, 810)
(123, 810)
123


In [10]:
# FILTERING / INCLUSION / EXCLUSION
print('Patients with less than 1 hour of sleep:')
print(np.sum(Xy_icu['h_sleep_SSComb1'] < 1))
print('EXCLUDE THESE PATIENTS.')

Xy_icu = Xy_icu.loc[Xy_icu.h_sleep_SSComb1 >= 1]


Patients with less than 1 hour of sleep:
10
EXCLUDE THESE PATIENTS.


In [14]:
stage_v = 'SSComb1'

In [69]:
bottom10_rem = Xy_icu.sort_values(by=[f'R{stage_v}']).iloc[:10].copy()
top10_rem = Xy_icu.sort_values(by=[f'R{stage_v}']).iloc[-10:].copy()

In [70]:
top10_rem = top10_rem[['study_id', 'MRN', 'Age', 'Sex', f'R{stage_v}', 'IBI_mean_SSComb1_S', 'IBI_std_SSComb1_S', 'MV_mean_SSComb1_S', 'MV_std_SSComb1_S', 'BSI_mean_SSComb1_S', 'BSI_std_SSComb1_S','RR_mean_SSComb1_S', 'RR_std_SSComb1_S']].round(2)
bottom10_rem = bottom10_rem[['study_id', 'MRN', 'Age', 'Sex', f'R{stage_v}', 'IBI_mean_SSComb1_S', 'IBI_std_SSComb1_S', 'MV_mean_SSComb1_S', 'MV_std_SSComb1_S', 'BSI_mean_SSComb1_S', 'BSI_std_SSComb1_S','RR_mean_SSComb1_S', 'RR_std_SSComb1_S']].round(2)

In [76]:
top10_rem.to_csv('top10_rem.csv', index=False)
bottom10_rem.to_csv('bottom10_rem.csv', index=False)

In [74]:
bottom10_rem

,study_id,MRN,Age,Sex,RSSComb1,IBI_mean_SSComb1_S,IBI_std_SSComb1_S,MV_mean_SSComb1_S,MV_std_SSComb1_S,BSI_mean_SSComb1_S,BSI_std_SSComb1_S,RR_mean_SSComb1_S,RR_std_SSComb1_S
308,icu_152,6751620.0,64.0,Male,0.0,2.88,0.88,0.09,0.06,0.11,0.08,23.57,3.71
358,icu_178,1745708.0,64.0,Female,0.0,2.82,0.93,0.14,0.08,0.15,0.08,23.72,4.67
119,icu_067,6603851.0,50.0,Female,0.0,3.46,0.82,0.12,0.08,0.13,0.08,19.55,3.32
171,icu_090,5946494.0,52.0,Female,0.0,3.41,1.26,0.14,0.09,0.17,0.08,20.92,3.99
176,icu_092,6546904.0,68.0,Female,0.0,2.12,0.58,0.11,0.09,0.13,0.08,28.75,3.63
101,icu_051,1810580.0,67.0,Male,0.0,3.92,1.55,0.13,0.07,0.17,0.06,19.57,4.38
85,icu_047,4352857.0,71.0,Male,0.0,3.77,0.70,0.11,0.07,0.10,0.06,17.56,3.08
304,icu_149,4203280.0,76.0,Female,0.0,4.74,1.57,0.16,0.12,0.16,0.12,13.86,2.38
99,icu_050,6577287.0,54.0,Female,0.0,5.08,1.04,0.13,0.07,0.12,0.06,15.29,2.84
249,icu_123,5761027.0,66.0,Male,0.0,5.28,3.85,0.51,0.22,0.45,0.13,14.14,5.32


In [75]:
top10_rem

,study_id,MRN,Age,Sex,RSSComb1,IBI_mean_SSComb1_S,IBI_std_SSComb1_S,MV_mean_SSComb1_S,MV_std_SSComb1_S,BSI_mean_SSComb1_S,BSI_std_SSComb1_S,RR_mean_SSComb1_S,RR_std_SSComb1_S
239,icu_119,6308196.0,62.0,Female,87.0,5.28,3.65,0.33,0.19,0.32,0.12,16.38,4.65
158,icu_084,6539323.0,62.0,Male,89.0,4.53,1.23,0.12,0.08,0.12,0.07,14.36,3.55
111,icu_061,5987312.0,54.0,Male,91.0,4.94,1.56,0.15,0.10,0.15,0.09,15.17,3.42
310,icu_153,480086.0,76.0,Male,91.0,2.25,0.61,0.12,0.08,0.14,0.08,27.72,3.65
63,icu_037,6492716.0,66.0,Male,92.0,4.07,1.49,0.16,0.10,0.19,0.10,18.14,3.89
247,icu_122,1194303.0,84.0,Male,95.0,4.26,0.81,0.12,0.07,0.11,0.06,16.53,3.20
192,icu_100,1723880.0,56.0,Male,97.0,2.95,1.15,0.14,0.07,0.19,0.07,22.94,3.77
341,icu_169,3165797.0,64.0,Male,99.0,3.65,0.92,0.08,0.06,0.10,0.06,17.33,3.01
161,icu_085,6358295.0,55.0,Female,100.0,4.17,1.19,0.14,0.07,0.16,0.07,16.71,2.55
71,icu_042,4101850.0,74.0,Female,100.0,3.76,1.20,0.14,0.10,0.16,0.09,17.81,3.32


In [102]:
# plt.figure()
# Xy_icu['h_sleep_SSComb1'].hist()
# plt.title('Cumulative Sleep (h) per Patient in ICU')
# plt.xlabel('Hours of Sleep')
# plt.ylabel('Number of Patients')
# plt.savefig('histogram_datacollection_hours_of_sleep_per_night.png', dpi=300)


plt.figure()
# .hist(bins=20)
plt.hist(Xy_icu['h_sleep_SSComb1'], bins=14, rwidth=0.95, color='navy')
plt.xlim([0,25])
plt.title('Amount of Sleep (h) per 24 Hours in the ICU')
plt.xlabel('Hours of Sleep')
plt.ylabel('Frequency (Number of Nights)')
plt.savefig('histogram_datacollection_hours_of_sleep_per_night.png', dpi=300)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:9: RuntimeWarning:

More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).



Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [103]:
# FILTERING / INCLUSION / EXCLUSION
print('Patients with less than 1 hour of sleep:')
print(np.sum(Xy_icu['h_sleep_SSComb1'] < 1))
print('EXCLUDE THESE PATIENTS.')

Xy_icu = Xy_icu.loc[Xy_icu.h_sleep_SSComb1 >= 1]


Patients with less than 1 hour of sleep:
0
EXCLUDE THESE PATIENTS.


In [104]:
def sleep_summary_barplots(ax, sel, variables, x_labels,  color='navy'):
    
    num = len(variables)
    ax.bar(np.arange(num), sel[variables].mean(axis=0), yerr=[np.zeros(num,), sel[variables].std(axis=0)], color=color)
    ax.set_xticks(np.arange(num))
    ax.set_xticklabels(x_labels, rotation=90)
    ax.set_title(f'ICU (n={len(pd.unique(sel.study_id))})')

    for iV in range(num):
        var_mean = np.round(sel[variables[iV]].mean(),1)
        if var_mean > 5:
            ax.annotate(str(var_mean), xy=(iV, var_mean-5), color='white', ha='center')
        else:
            ax.annotate(str(var_mean), xy=(iV, var_mean+10), color='black', ha='center')
    return ax

stage_v = 'SSComb1'

variables = [f'h_sleep_{stage_v}', f'N1{stage_v}',
       f'N2{stage_v}', f'N3{stage_v}', f'R{stage_v}', f'SFI{stage_v}', f'ArousalIndex{stage_v}']
x_labels = ['Sleep (h)', 'N1 (%)',
       'N2 (%)', 'N3 (%)', 'R (%)', 'SFI', 'Arousal I.']

fig, (ax1, ax2) = plt.subplots(1,2, figsize=(9,3), sharey=True)

sel = Xy_icu
ax1 = sleep_summary_barplots(ax1, sel, variables, x_labels,  color='navy')
ax1.set_title(f'ICU (n={len(pd.unique(sel.study_id))})')
sel = Xy_sleeplab
ax2 = sleep_summary_barplots(ax2, sel, variables, x_labels,  color='goldenrod')
ax2.set_title(f'Sleeplab (n={len(pd.unique(sel.study_id))})')
plt.subplots_adjust(bottom=0.27)
plt.tight_layout()
plt.savefig('sleep_stage_distribution.png', dpi=300)

fig, (ax1, ax2, ax3) = plt.subplots(1,3, figsize=(12,3), sharey=True)

stage_v = 'SSComb1'
variables = [f'h_sleep_{stage_v}', f'N1{stage_v}',
       f'N2{stage_v}', f'N3{stage_v}', f'R{stage_v}', f'SFI{stage_v}', f'ArousalIndex{stage_v}']
x_labels = ['Sleep (h)', 'N1 (%)',
       'N2 (%)', 'N3 (%)', 'R (%)', 'SFI', 'Arousal I.']

sel = Xy_icu
ax1 = sleep_summary_barplots(ax1, sel, variables, x_labels,  color='navy')
ax1.set_title(f'ICU (n={len(pd.unique(sel.study_id))})')
sel = Xy_sleeplab
ax2 = sleep_summary_barplots(ax2, sel, variables, x_labels,  color='goldenrod')
ax2.set_title(f'Sleeplab (n={len(pd.unique(sel.study_id))}), Model')

stage_v = 'SSExpert'
variables = [f'h_sleep_{stage_v}', f'N1{stage_v}',
       f'N2{stage_v}', f'N3{stage_v}', f'R{stage_v}', f'SFI{stage_v}', f'ArousalIndex{stage_v}']
x_labels = ['Sleep (h)', 'N1 (%)',
       'N2 (%)', 'N3 (%)', 'R (%)', 'SFI', 'Arousal I.']
ax3 = sleep_summary_barplots(ax3, sel, variables, x_labels,  color='forestgreen')
ax3.set_title(f'Sleeplab (n={len(pd.unique(sel.study_id))}), Expert')

plt.subplots_adjust(bottom=0.27)
plt.tight_layout()

plt.savefig('sleep_stage_distribution_wexpert.png', dpi=300)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:24: RuntimeWarning:

More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).



Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:36: RuntimeWarning:

More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).



Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [121]:
# plt.figure()
# Xy_icu['h_sleep_SSComb1'].hist()
# plt.title('Cumulative Sleep (h) per Patient in ICU')
# plt.xlabel('Hours of Sleep')
# plt.ylabel('Number of Patients')
# plt.savefig('histogram_datacollection_hours_of_sleep_per_night.png', dpi=300)

stage_v = 'SSComb1'

variables = [f'h_sleep_{stage_v}', f'N1{stage_v}',
       f'N2{stage_v}', f'N3{stage_v}', f'R{stage_v}', f'SFI{stage_v}', f'ArousalIndex{stage_v}']
x_labels = ['Sleep (h)', 'N1 (%)',
       'N2 (%)', 'N3 (%)', 'R (%)', 'SFI', 'Arousal I.']


fig3 = plt.figure(constrained_layout=True, figsize=(7,7))
gs = fig3.add_gridspec(2, 2, height_ratios=(1,2))
f3_ax1 = fig3.add_subplot(gs[0, :])

f3_ax1.hist(Xy_icu['h_sleep_SSComb1'], bins=14, rwidth=0.95, color='navy')
f3_ax1.set_xlim([0,25])
f3_ax1.set_title('Amount of Sleep (h) per 24 Hours in the ICU')
f3_ax1.set_xlabel('Hours of Sleep')
f3_ax1.set_ylabel('Frequency (Number of Nights)')

f3_ax2 = fig3.add_subplot(gs[1, 0])
sel = Xy_icu
f3_ax2 = sleep_summary_barplots(f3_ax2, sel, variables, x_labels,  color='navy')

f3_ax2.set_title(f'Sleep Stages ICU (n={len(pd.unique(sel.study_id))})')
f3_ax3 = fig3.add_subplot(gs[1, 1])
sel = Xy_sleeplab
f3_ax3 = sleep_summary_barplots(f3_ax3, sel, variables, x_labels,  color='goldenrod')
f3_ax3.set_title(f'Sleep Stages Sleeplab (n={len(pd.unique(sel.study_id))})')



# sel = Xy_sleeplab
# ax2.set_title(f'Sleeplab (n={len(pd.unique(sel.study_id))})')
# plt.subplots_adjust(bottom=0.27)
# plt.tight_layout()


# # .hist(bins=20)
# ax3.hist(Xy_icu['h_sleep_SSComb1'], bins=14, rwidth=0.95, color='navy')
# ax3.set_xlim([0,25])
# ax3.set_title('Amount of Sleep (h) per 24 Hours in the ICU')
# ax3.set_xlabel('Hours of Sleep')
# ax3.set_ylabel('Frequency (Number of Nights)')

plt.savefig('histogram_and_sleepstages.png', dpi=300)
plt.savefig('histogram_and_sleepstages.svg', dpi=300)
plt.savefig('histogram_and_sleepstages.pdf', dpi=300)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:16: RuntimeWarning:

More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).



Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [64]:
stage_v = 'SSComb1'

variables = [f'h_sleep_{stage_v}', f'N1{stage_v}',
       f'N2N3{stage_v}', f'R{stage_v}', f'SFI{stage_v}', f'ArousalIndex{stage_v}']
x_labels = ['Sleep (h)', 'N1 (%)',
       'N2+N3 (%)', 'R (%)', 'SFI', 'Arousal I.']

fig, (ax1, ax2) = plt.subplots(1,2, figsize=(9,3), sharey=True)

sel = Xy_icu
ax1 = sleep_summary_barplots(ax1, sel, variables, x_labels,  color='navy')
ax1.set_title(f'ICU (n={len(pd.unique(sel.study_id))})')
sel = Xy_sleeplab
ax2 = sleep_summary_barplots(ax2, sel, variables, x_labels,  color='goldenrod')
ax2.set_title(f'Sleeplab (n={len(pd.unique(sel.study_id))})')
plt.subplots_adjust(bottom=0.3)
plt.savefig('sleep_stage_distribution_N2N3.png', dpi=300)


fig, (ax1, ax2, ax3) = plt.subplots(1,3, figsize=(13,3), sharey=True)

stage_v = 'SSComb1'
variables = [f'h_sleep_{stage_v}', f'N1{stage_v}',
       f'N2N3{stage_v}', f'R{stage_v}', f'SFI{stage_v}', f'ArousalIndex{stage_v}']
x_labels = ['Sleep (h)', 'N1 (%)',
       'N2+N3 (%)', 'R (%)', 'SFI', 'Arousal I.']
sel = Xy_icu
ax1 = sleep_summary_barplots(ax1, sel, variables, x_labels,  color='navy')
ax1.set_title(f'ICU (n={len(pd.unique(sel.study_id))})')
sel = Xy_sleeplab
ax2 = sleep_summary_barplots(ax2, sel, variables, x_labels,  color='goldenrod')
ax2.set_title(f'Sleeplab (n={len(pd.unique(sel.study_id))}), Model')

stage_v = 'SSExpert'
variables = [f'h_sleep_{stage_v}', f'N1{stage_v}',
       f'N2N3{stage_v}', f'R{stage_v}', f'SFI{stage_v}', f'ArousalIndex{stage_v}']
x_labels = ['Sleep (h)', 'N1 (%)',
       'N2+N3 (%)', 'R (%)', 'SFI', 'Arousal I.']
ax3 = sleep_summary_barplots(ax3, sel, variables, x_labels,  color='forestgreen')
ax3.set_title(f'Sleeplab (n={len(pd.unique(sel.study_id))}), Expert')


plt.subplots_adjust(bottom=0.3)

plt.savefig('sleep_stage_distribution_N2N3_wexpert.png', dpi=300)


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:8: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:20: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [103]:
sns.scatterplot

<function seaborn.relational.scatterplot(x=None, y=None, hue=None, style=None, size=None, data=None, palette=None, hue_order=None, hue_norm=None, sizes=None, size_order=None, size_norm=None, markers=True, style_order=None, x_bins=None, y_bins=None, units=None, estimator=None, ci=95, n_boot=1000, alpha='auto', x_jitter=None, y_jitter=None, legend='brief', ax=None, **kwargs)>

In [65]:
for i in range(100):
    plt.close()

In [22]:

def default_scatter():
    plt.figure(figsize=figsize)
    plt.scatter(Xy_sleeplab[x_var], Xy_sleeplab[y_var], c='goldenrod', s=s, alpha=alpha)
    plt.scatter(Xy_icu[x_var], Xy_icu[y_var], c='navy', s=s, alpha=alpha)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend(['Sleeplab', 'ICU'])
    plt.title(title)
    plt.tight_layout()
    
    
def default_scatter_ax0(ax):
    ax.scatter(Xy_sleeplab[x_var], Xy_sleeplab[y_var], c='goldenrod', s=s, alpha=alpha)
    ax.scatter(Xy_icu[x_var], Xy_icu[y_var], c='navy', s=s, alpha=alpha)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(['Sleeplab', 'ICU'])
    ax.set_title(title)
    
def default_scatter_ax(ax, x_var, y_var,title):
    ax.scatter(Xy_sleeplab[x_var], Xy_sleeplab[y_var], c='goldenrod', s=s, alpha=alpha)
    ax.scatter(Xy_icu[x_var], Xy_icu[y_var], c='navy', s=s, alpha=alpha)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(['Sleeplab', 'ICU'])
    ax.set_title(title)
    



def grid_plot1():       

    fig, ax = plt.subplots(2,3, figsize=figsize_grid, sharex=True, sharey=True)


    x_var = f'{variable_name}_mean'
    y_var = f'{variable_name}_std'
    s = 7
    alpha = 0.8
    title = f'{variable_short} total' #  f'{variable_long} ({variable_short}) total'
    default_scatter_ax(ax[0,0], x_var, y_var, title)

    x_var = f'{variable_name}_mean_SSComb1_W'
    y_var = f'{variable_name}_std_SSComb1_W'
    s = 7
    alpha = 0.8
    title = f'{variable_short} in Wake'
    default_scatter_ax(ax[0,1], x_var, y_var, title)

    x_var = f'{variable_name}_mean_SSComb1_S'
    y_var = f'{variable_name}_std_SSComb1_S'
    s = 7
    alpha = 0.8
    title = f'{variable_short} in Sleep'
    default_scatter_ax(ax[0,2], x_var, y_var, title)

    x_var = f'{variable_name}_mean_SSComb1_N2N3'
    y_var = f'{variable_name}_std_SSComb1_N2N3'
    s = 7
    alpha = 0.8
    title = f'{variable_short} in N2 and N3'
    default_scatter_ax(ax[1,0], x_var, y_var, title)

    x_var = f'{variable_name}_mean_SSComb1_N1'
    y_var = f'{variable_name}_std_SSComb1_N1'
    s = 7
    alpha = 0.8
    title = f'{variable_short} in N1'
    default_scatter_ax(ax[1,1], x_var, y_var, title)

    x_var = f'{variable_name}_mean_SSComb1_R'
    y_var = f'{variable_name}_std_SSComb1_R'

    title = f'{variable_short} in R'
    default_scatter_ax(ax[1,2], x_var, y_var, title)


    for ax0_tmp, ax1_tmp in ([0,1], [0,2], [1,0], [1,1],[1,2]):
        ax[ax0_tmp, ax1_tmp].legend(frameon=False)

    for ax0_tmp, ax1_tmp in ([0,1], [0,2], [0,0]):
        ax[ax0_tmp, ax1_tmp].set_xlabel('')

    for ax0_tmp, ax1_tmp in ([0,1], [0,2], [1,1], [1,2]):
        ax[ax0_tmp, ax1_tmp].set_ylabel('')
    
    plt.tight_layout()
    
    
    


def grid_plot2():       

    fig, ax = plt.subplots(1,2, figsize=(5,3), sharex=True, sharey=True)


    x_var = f'{variable_name}_mean_SSComb1_W'
    y_var = f'{variable_name}_std_SSComb1_W'
    s = 7
    alpha = 0.8
    title = f'{variable_short} in Wake'
    default_scatter_ax(ax[0], x_var, y_var, title)

    x_var = f'{variable_name}_mean_SSComb1_S'
    y_var = f'{variable_name}_std_SSComb1_S'
    s = 7
    alpha = 0.8
    title = f'{variable_short} in Sleep'
    default_scatter_ax(ax[1], x_var, y_var, title)


#     for ax0_tmp, ax1_tmp in ([0,1], [0,2], [1,0], [1,1],[1,2]):
#         ax[ax0_tmp, ax1_tmp].legend(frameon=False)

#     for ax0_tmp, ax1_tmp in ([0,1], [0,2], [0,0]):
#         ax[ax0_tmp, ax1_tmp].set_xlabel('')

#     for ax0_tmp, ax1_tmp in ([0,1], [0,2], [1,1], [1,2]):
#         ax[ax0_tmp, ax1_tmp].set_ylabel('')
    
    plt.tight_layout()
    
    
    

def grid_plot1_median():       

    fig, ax = plt.subplots(2,3, figsize=(9,6), sharex=True, sharey=True)


    x_var = f'{variable_name}_median'
    y_var = f'{variable_name}_IQR'
    s = 7
    alpha = 0.8
    title = f'{variable_long} ({variable_short}) total'
    default_scatter_ax(ax[0,0], x_var, y_var, title)

    x_var = f'{variable_name}_median_SSComb1_W'
    y_var = f'{variable_name}_IQR_SSComb1_W'
    s = 7
    alpha = 0.8
    title = f'{variable_short} in Wake'
    default_scatter_ax(ax[0,1], x_var, y_var, title)

    x_var = f'{variable_name}_median_SSComb1_S'
    y_var = f'{variable_name}_IQR_SSComb1_S'
    s = 7
    alpha = 0.8
    title = f'{variable_short} in Sleep'
    default_scatter_ax(ax[0,2], x_var, y_var, title)

    x_var = f'{variable_name}_median_SSComb1_N2N3'
    y_var = f'{variable_name}_IQR_SSComb1_N2N3'
    s = 7
    alpha = 0.8
    title = f'{variable_short} in N2 and N3'
    default_scatter_ax(ax[1,0], x_var, y_var, title)

    x_var = f'{variable_name}_median_SSComb1_N1'
    y_var = f'{variable_name}_IQR_SSComb1_N1'
    s = 7
    alpha = 0.8
    title = f'{variable_short} in N1'
    default_scatter_ax(ax[1,1], x_var, y_var, title)

    x_var = f'{variable_name}_median_SSComb1_R'
    y_var = f'{variable_name}_IQR_SSComb1_R'

    title = f'{variable_short} in R'
    default_scatter_ax(ax[1,2], x_var, y_var, title)


    for ax0_tmp, ax1_tmp in ([0,1], [0,2], [1,0], [1,1],[1,2]):
        ax[ax0_tmp, ax1_tmp].legend(frameon=False)

    for ax0_tmp, ax1_tmp in ([0,1], [0,2], [0,0]):
        ax[ax0_tmp, ax1_tmp].set_xlabel('')

    for ax0_tmp, ax1_tmp in ([0,1], [0,2], [1,1], [1,2]):
        ax[ax0_tmp, ax1_tmp].set_ylabel('')
    
    plt.tight_layout()
    

In [12]:
Xy_sleeplab['study_night_id'] = Xy_sleeplab.study_id.apply(lambda x: x.replace('sleeplab', 'SL')) + ' ' + Xy_sleeplab.night_id
# Xy_sleeplab.study_id.values;
# Xy_sleeplab.night_id.values

In [13]:
Xy_icu['study_night_id'] = Xy_icu.study_id + ' ' + Xy_icu.night_id


In [14]:
import plotly # built with plotly version '4.4.1'
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
from plotly.subplots import make_subplots


def default_scatter_plotly(x_var, y_var, title, showlegend):
    
    trace_sleeplab = go.Scatter(x=Xy_sleeplab[x_var].round(3), y=Xy_sleeplab[y_var].round(3), hoverinfo = 'x+y+text', 
                             hovertext = Xy_sleeplab['study_night_id'],  fillcolor='goldenrod', mode='markers', 
                       marker=dict(size=s, color='goldenrod'), opacity=1, name='Sleeplab', showlegend=showlegend)
                             
    trace_icu = go.Scatter(x=Xy_icu[x_var].round(3), y=Xy_icu[y_var].round(3), hoverinfo = 'x+y+text', 
                             hovertext = Xy_icu['study_night_id'],  fillcolor='goldenrod', mode='markers', 
                       marker=dict(size=s, color='navy'), opacity=1, name='ICU', showlegend=showlegend)                             

    return trace_sleeplab, trace_icu
    
    
    
def grid_plot1_plotly():   
    
    
    fig = make_subplots(rows=2, cols=3, shared_xaxes=True, shared_yaxes=True, vertical_spacing=0.05,
                           subplot_titles=(f'{variable_long} ({variable_short}) total', f'{variable_short} in Wake',
                   f'{variable_short} in Sleep', f'{variable_short} in N2 and N3',
                   f'{variable_short} in N1', f'{variable_short} in R')                       
                       )

    s = 7
    alpha = 0.8
    
    showlegend=True
    x_var = f'{variable_name}_mean'
    y_var = f'{variable_name}_std'
    title = f'{variable_long} ({variable_short}) total'
    trace_sleeplab, trace_icu = default_scatter_plotly(x_var, y_var, title, showlegend)
    fig.add_trace(trace_sleeplab, 1, 1)
    fig.add_trace(trace_icu, 1, 1)
    
    showlegend=False

    x_var = f'{variable_name}_mean_SSComb1_W'
    y_var = f'{variable_name}_std_SSComb1_W'
    title = f'{variable_short} in Wake'
    trace_sleeplab, trace_icu = default_scatter_plotly(x_var, y_var, title, showlegend)
    fig.add_trace(trace_sleeplab, 1, 2)
    fig.add_trace(trace_icu, 1, 2)
    
    x_var = f'{variable_name}_mean_SSComb1_S'
    y_var = f'{variable_name}_std_SSComb1_S'
    title = f'{variable_short} in Sleep'
    trace_sleeplab, trace_icu = default_scatter_plotly(x_var, y_var, title, showlegend)
    fig.add_trace(trace_sleeplab, 1, 3)
    fig.add_trace(trace_icu, 1, 3)

    x_var = f'{variable_name}_mean_SSComb1_N2N3'
    y_var = f'{variable_name}_std_SSComb1_N2N3'
    title = f'{variable_short} in N2 and N3'
    trace_sleeplab, trace_icu = default_scatter_plotly(x_var, y_var, title, showlegend)
    fig.add_trace(trace_sleeplab, 2, 1)
    fig.add_trace(trace_icu, 2, 1)

    x_var = f'{variable_name}_mean_SSComb1_N1'
    y_var = f'{variable_name}_std_SSComb1_N1'
    title = f'{variable_short} in N1'
    trace_sleeplab, trace_icu = default_scatter_plotly(x_var, y_var, title, showlegend)
    fig.add_trace(trace_sleeplab, 2, 2)
    fig.add_trace(trace_icu, 2, 2)

    x_var = f'{variable_name}_mean_SSComb1_R'
    y_var = f'{variable_name}_std_SSComb1_R'
    title = f'{variable_short} in R'
    trace_sleeplab, trace_icu = default_scatter_plotly(x_var, y_var, title, showlegend)
    fig.add_trace(trace_sleeplab, 2, 3)
    fig.add_trace(trace_icu, 2, 3)

    plot(fig, filename=f'Scatter_ICU_Sleeplab_{variable_long}.html', auto_open=auto_open)
    

In [15]:
variable_name = 'BSI'

In [16]:
[x for x in Xy.columns if 'median' in x];

In [36]:
print('Total, sleep and wake"')

for variable_name in ['IBI', 'MV', 'BSI']:
    print('')
    print(variable_name)
    print(f"Sleeplab   Mean: {Xy_sleeplab[f'{variable_name}_mean'].mean().round(2)} +- {Xy_sleeplab[f'{variable_name}_mean'].std().round(2)}, STD:{Xy_sleeplab[f'{variable_name}_std'].mean().round(2)} +- {Xy_sleeplab[f'{variable_name}_std'].std().round(2)}")
    print(f"ICU        Mean: {Xy_icu[f'{variable_name}_mean'].mean().round(2)} +- {Xy_icu[f'{variable_name}_mean'].std().round(2)}, STD:{Xy_icu[f'{variable_name}_std'].mean().round(2)} +- {Xy_icu[f'{variable_name}_std'].std().round(2)}")

print('____________________________________________________')
print('')
print('Sleep Only')

    
for variable_name in ['IBI', 'MV', 'BSI', 'RR']:
    print('')
    print(variable_name)
    print(f"Sleeplab   Mean: {Xy_sleeplab[f'{variable_name}_mean_SSComb1_S'].mean().round(2)} +- {Xy_sleeplab[f'{variable_name}_mean_SSComb1_S'].std().round(2)}, STD:{Xy_sleeplab[f'{variable_name}_std_SSComb1_S'].mean().round(2)} +- {Xy_sleeplab[f'{variable_name}_std_SSComb1_S'].std().round(2)}")
    print(f"ICU        Mean: {Xy_icu[f'{variable_name}_mean_SSComb1_S'].mean().round(2)} +- {Xy_icu[f'{variable_name}_mean_SSComb1_S'].std().round(2)}, STD:{Xy_icu[f'{variable_name}_std_SSComb1_S'].mean().round(2)} +- {Xy_icu[f'{variable_name}_std_SSComb1_S'].std().round(2)}")
    

Total, sleep and wake"

IBI
Sleeplab   Mean: 4.5 +- 0.72, STD:2.11 +- 0.69
ICU        Mean: 3.62 +- 0.83, STD:1.71 +- 0.72

MV
Sleeplab   Mean: 0.26 +- 0.09, STD:0.22 +- 0.07
ICU        Mean: 0.19 +- 0.08, STD:0.14 +- 0.06

BSI
Sleeplab   Mean: 0.22 +- 0.07, STD:0.15 +- 0.03
ICU        Mean: 0.2 +- 0.06, STD:0.12 +- 0.03
____________________________________________________

Sleep Only

IBI
Sleeplab   Mean: 4.42 +- 0.73, STD:1.76 +- 0.63
ICU        Mean: 3.63 +- 0.88, STD:1.35 +- 0.68

MV
Sleeplab   Mean: 0.23 +- 0.08, STD:0.18 +- 0.06
ICU        Mean: 0.17 +- 0.08, STD:0.11 +- 0.05

BSI
Sleeplab   Mean: 0.2 +- 0.06, STD:0.12 +- 0.03
ICU        Mean: 0.18 +- 0.07, STD:0.09 +- 0.03

RR
Sleeplab   Mean: 16.02 +- 2.25, STD:3.27 +- 0.84
ICU        Mean: 19.66 +- 4.08, STD:3.59 +- 1.04


In [72]:
s = 7
alpha = 0.8
auto_open = True

variable_long = 'Inter-Breath-Interval'
variable_short = 'IBI'
variable_name = 'IBI'
xlabel = 'Mean (s)'
ylabel =  'STD (s)'

grid_plot1_plotly()

variable_long = 'Breathing Variability Index'
variable_short = 'BVI'
variable_name = 'BSI'
xlabel = 'Mean'
ylabel =  'STD'
grid_plot1_plotly()

variable_long = 'Minute Ventilation Coef.Var.'
variable_short = 'MV'
variable_name = 'MV'
xlabel = 'Mean'
ylabel =  'STD'
grid_plot1_plotly()

variable_long = 'Respiratory Rate'
variable_short = 'RR'
variable_name = 'RR'
xlabel = 'Mean (bpm)'
ylabel =  'STD (bpm)'
grid_plot1_plotly()



In [23]:
s = 7
alpha = 0.8
figsize_grid = (6,4)

variable_long = 'Inter-Breath-Interval'
variable_long = 'Inter-Breath-Interval'
variable_short = 'IBI'
variable_name = 'IBI'
xlabel = 'Mean (s)'
ylabel =  'STD (s)'
add_on = '_small'

grid_plot1()
plt.tight_layout()
plt.savefig(f'Scatter_ICU_Sleeplab_InterBreathInterval_nights{add_on}.png', dpi=300)

variable_long = 'Breathing Variability Index'
variable_short = 'BVI'
variable_name = 'BSI'
xlabel = 'Mean'
ylabel =  'STD'

grid_plot1()
plt.savefig(f'Scatter_ICU_Sleeplab_BreathingVariability_nights{add_on}.png', dpi=300)

variable_long = 'Minute Ventilation Coef.Var.'
variable_short = 'MV'
variable_name = 'MV'
xlabel = 'Mean'
ylabel =  'STD'

grid_plot1()
plt.savefig(f'Scatter_ICU_Sleeplab_MinuteVentilation_nights{add_on}.png', dpi=300)

# variable_long = 'Respiratory Rate'
# variable_short = 'RR'
# variable_name = 'RR'
# xlabel = 'Mean (bpm)'
# ylabel =  'STD (bpm)'

# grid_plot1()
# plt.savefig('Scatter_ICU_Sleeplab_RespiratoryRate_nights.png', dpi=300)

variable_long = 'Respiratory Rate'
variable_short = 'RR'
variable_name = 'RR'
xlabel = 'Mean (cpm)'
ylabel =  'STD (cpm)'

grid_plot2()
plt.savefig(f'Scatter_ICU_Sleeplab_RespiratoryRate_nights{add_on}.png', dpi=300)


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

No handles with labels found to put in legend.
No handles with labels found to put in legend.
No handles with labels found to put in legend.
No handles with labels found to put in legend.
No handles with labels found to put in legend.


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

No handles with labels found to put in legend.
No handles with labels found to put in legend.
No handles with labels found to put in legend.
No handles with labels found to put in legend.
No handles with labels found to put in legend.


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

No handles with labels found to put in legend.
No handles with labels found to put in legend.
No handles with labels found to put in legend.
No handles with labels found to put in legend.
No handles with labels found to put in legend.


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [230]:
fig, ax = plt.subplots(2,2, figsize=(7,7), sharex=True, sharey=True)

x_var = 'BSI_mean'
y_var = 'IBI_std'
s = 7
alpha = 0.8
title = 'Breathing Stability Index (BSI) total'
xlabel = ' Mean (s)'
ylabel =  'STD (s)'
figsize = (5,3)
default_scatter_ax(ax[0,0])

x_var = 'BSI_mean_SSComb1_W'
y_var = 'BSI_std_SSComb1_W'
s = 7
alpha = 0.8
title = 'BSI in Wake'
xlabel = ' Mean (s)'
ylabel =  'STD (s)'
figsize = (5,3)
default_scatter_ax(ax[0,1])


x_var = 'BSI_mean_SSComb1_S'
y_var = 'BSI_std_SSComb1_S'
s = 7
alpha = 0.8
title = 'BSI in Sleep'
xlabel = ' Mean'
ylabel =  'STD'
figsize = (5,3)
default_scatter_ax(ax[0,0])

x_var = 'BSI_mean_SSComb1_N2N3'
y_var = 'BSI_std_SSComb1_N2N3'
s = 7
alpha = 0.8
title = 'BSI in N2 and N3'
figsize = (5,3)
default_scatter_ax(ax[0,1])

x_var = 'BSI_mean_SSComb1_N1'
y_var = 'BSI_std_SSComb1_N1'
s = 7
alpha = 0.8
title = 'BSI in N1'
figsize = (5,3)
default_scatter_ax(ax[1,0])

x_var = 'BSI_mean_SSComb1_R'
y_var = 'BSI_std_SSComb1_R'
s = 7
alpha = 0.8
title = 'BSI in R'
figsize = (5,3)
default_scatter_ax(ax[1,1])

ax[0,0].set_xlabel('')
ax[0,1].set_xlabel('')
ax[0,1].set_ylabel('')
ax[1,1].set_ylabel('')
ax[0,1].legend(frameon=False)
ax[1,0].legend(frameon=False)
ax[1,1].legend(frameon=False)

plt.tight_layout()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

No handles with labels found to put in legend.
No handles with labels found to put in legend.
No handles with labels found to put in legend.


In [231]:
[x for x in Xy.columns if 'RR_mean' in x]

['RR_mean',
 'RR_mean_SSResp_N1',
 'RR_mean_SSResp_N2',
 'RR_mean_SSResp_N3',
 'RR_mean_SSResp_N2N3',
 'RR_mean_SSResp_NR',
 'RR_mean_SSResp_R',
 'RR_mean_SSResp_W',
 'RR_mean_SSComb1_N1',
 'RR_mean_SSComb1_N2',
 'RR_mean_SSComb1_N3',
 'RR_mean_SSComb1_N2N3',
 'RR_mean_SSComb1_NR',
 'RR_mean_SSComb1_R',
 'RR_mean_SSComb1_W',
 'RR_mean_SSExpert_N1',
 'RR_mean_SSExpert_N2',
 'RR_mean_SSExpert_N3',
 'RR_mean_SSExpert_N2N3',
 'RR_mean_SSExpert_NR',
 'RR_mean_SSExpert_R',
 'RR_mean_SSExpert_W',
 'RR_mean_SSResp_S',
 'RR_mean_SSComb1_S',
 'RR_mean_SSExpert_S']

In [190]:
fig, ax = plt.subplots(2,2, figsize=(7,7), sharex=True, sharey=True)
var = 'RR'
x_var = f'{var}_mean'
y_var = f'{var}_std'
s = 7
alpha = 0.8
title = 'Respiratory Rate (RR)'
xlabel = ' Mean'
ylabel =  'STD'
figsize = (5,3)
default_scatter_ax(ax[0,0])

x_var = f'{var}_mean_SSComb1_N2N3'
y_var = f'{var}_std_SSComb1_N2N3'
s = 7
alpha = 0.8
title = 'RR in N2 and N3'
figsize = (5,3)
default_scatter_ax(ax[0,1])

x_var = f'{var}_mean_SSComb1_N1'
y_var = f'{var}_std_SSComb1_N1'
s = 7
alpha = 0.8
title = 'RR in N1'
figsize = (5,3)
default_scatter_ax(ax[1,0])

x_var = f'{var}_mean_SSComb1_R'
y_var = f'{var}_std_SSComb1_R'
s = 7
alpha = 0.8
title = 'RR in R'
figsize = (5,3)
default_scatter_ax(ax[1,1])

ax[0,0].set_xlabel('')
ax[0,1].set_xlabel('')
ax[0,1].set_ylabel('')
ax[1,1].set_ylabel('')
ax[0,1].legend(frameon=False)
ax[1,0].legend(frameon=False)
ax[1,1].legend(frameon=False)

plt.tight_layout()

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:1: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  """Entry point for launching an IPython kernel.


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

No handles with labels found to put in legend.
No handles with labels found to put in legend.
No handles with labels found to put in legend.


In [192]:
fig, ax = plt.subplots(2,2, figsize=(7,7), sharex=True, sharey=True)
var = 'MV'
x_var = f'{var}_mean'
y_var = f'{var}_std'
s = 7
alpha = 0.8
title = 'Minute Ventilation (MV)'
xlabel = ' Mean'
ylabel =  'STD'
figsize = (5,3)
default_scatter_ax(ax[0,0])

x_var = f'{var}_mean_SSComb1_N2N3'
y_var = f'{var}_std_SSComb1_N2N3'
s = 7
alpha = 0.8
title = 'MV in N2 and N3'
figsize = (5,3)
default_scatter_ax(ax[0,1])

x_var = f'{var}_mean_SSComb1_N1'
y_var = f'{var}_std_SSComb1_N1'
s = 7
alpha = 0.8
title = 'MV in N1'
figsize = (5,3)
default_scatter_ax(ax[1,0])

x_var = f'{var}_mean_SSComb1_R'
y_var = f'{var}_std_SSComb1_R'
s = 7
alpha = 0.8
title = 'MV in R'
figsize = (5,3)
default_scatter_ax(ax[1,1])

ax[0,0].set_xlabel('')
ax[0,1].set_xlabel('')
ax[0,1].set_ylabel('')
ax[1,1].set_ylabel('')
ax[0,1].legend(frameon=False)
ax[1,0].legend(frameon=False)
ax[1,1].legend(frameon=False)

plt.tight_layout()

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:1: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  """Entry point for launching an IPython kernel.


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

No handles with labels found to put in legend.
No handles with labels found to put in legend.
No handles with labels found to put in legend.


In [118]:
pd.unique(Xy.population)

array(['icu', 'sleeplab'], dtype=object)

In [106]:
Xy

,study_id,night_id,population,MRN,Age,Sex,IBI_mean,IBI_std,IBI_median,IBI_Q75,...,N1SSExpert,N2SSExpert,N3SSExpert,RSSExpert,SFISSExpert,SFI2SSExpert,ArousalIndexSSExpert,N2N3SSResp,N2N3SSComb1,N2N3SSExpert
192,icu_100,2019-06-19,icu,1723880.0,56.0,Male,2.867059,0.936676,2.800781,3.199219,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,3.0,NaN
194,icu_101,2019-06-24,icu,6676668.0,71.0,Male,2.034358,1.587225,1.599609,1.900391,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80.0,87.0,NaN
196,icu_102,2019-06-25,icu,2390500.0,66.0,Male,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
198,icu_103,2019-06-26,icu,4981686.0,68.0,Male,2.454354,0.965093,2.300781,2.599609,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53.0,49.0,NaN
201,icu_104,2019-06-26,icu,5444546.0,64.0,Male,3.624557,1.146319,3.599609,4.101562,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,40.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
798,sleeplab_458,2020-01-11,sleeplab,3544572.0,69.0,Male,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
799,sleeplab_460,2020-01-11,sleeplab,4567135.0,56.1,Male,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
800,sleeplab_461,2020-01-12,sleeplab,3619992.0,64.5,Male,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
803,sleeplab_464,2020-01-13,sleeplab,4777940.0,69.2,Male,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
